# PSF Extraction Tester
Uses calexp images — same Butler pattern as the official DP0.2 tutorial.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

print(f'CWD  : {os.getcwd()}')
print(f'USER : {os.environ.get("USER", "unknown")}')

In [ ]:
# Exact pattern from the official DP0.2 tutorial
from lsst.daf.butler import Butler
from lsst.geom import Point2D

butler = Butler('dp02', collections='2.2i/runs/DP0.2')
print('Butler connected')

In [ ]:
# Find calexp images — same as tutorial
tract = 3828
where = f"instrument='LSSTCam-imSim' AND skymap='DC2' AND tract={tract} AND detector=19 AND band='i'"
calexp_refs = sorted(list(set(butler.query_datasets('calexp', where=where))))
print(f'Found {len(calexp_refs)} calexps')

dataId = calexp_refs[5].dataId
print(f'Using: {dataId}')

calexp = butler.get('calexp', dataId=dataId)
image  = calexp.image.array
ny, nx = image.shape
psf    = calexp.getPsf()
wcs    = calexp.getWcs()
PIXEL_SCALE = wcs.getPixelScale().asArcseconds()

print(f'Image shape  : {image.shape}')
print(f'PSF type     : {type(psf).__name__}')
print(f'Pixel scale  : {PIXEL_SCALE:.4f} arcsec/px')

## Scan for valid PSF positions

In [ ]:
# Test center first
center = Point2D(float(nx//2), float(ny//2))
try:
    psf.computeImage(center)
    print(f'Center ({nx//2}, {ny//2}): OK')
except Exception as e:
    print(f'Center ({nx//2}, {ny//2}): FAILED — {e}')

In [ ]:
# Scan grid
margin, n_side = 200, 5
xs = np.linspace(margin, nx-margin, n_side)
ys = np.linspace(margin, ny-margin, n_side)

results, failures = [], []
for y in ys:
    for x in xs:
        pos = Point2D(float(x), float(y))
        try:
            arr = psf.computeImage(pos).array.copy()
            sh  = psf.computeShape(pos)
            fw  = 2.355 * np.sqrt((sh.getIxx() + sh.getIyy()) / 2.0)
            results.append({'x': x, 'y': y, 'fwhm': fw,
                            'ixx': sh.getIxx(), 'iyy': sh.getIyy(), 'ixy': sh.getIxy(),
                            'kernel_shape': arr.shape, 'kernel_sum': arr.sum(), 'psf': arr})
        except Exception as e:
            failures.append({'x': x, 'y': y, 'error': str(e)})

fwhms = [r['fwhm'] for r in results]
print(f'Succeeded: {len(results)}/{n_side**2}')
print(f'Failed   : {len(failures)}')
if results:
    print(f'FWHM mean : {np.mean(fwhms):.4f} px = {np.mean(fwhms)*PIXEL_SCALE:.4f} arcsec')
    print(f'FWHM range: [{min(fwhms):.4f}, {max(fwhms):.4f}] px')
for f in failures[:3]:
    print(f"  FAIL ({f['x']:.0f},{f['y']:.0f}): {f['error'][:60]}")

In [ ]:
# PSF grid plot
n_show = min(len(results), n_side*n_side)
n_cols = min(n_side, n_show)
n_rows = int(np.ceil(n_show/n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5*n_cols, 2.5*n_rows))
for idx, ax in enumerate(np.array(axes).flat):
    if idx < len(results):
        r = results[idx]
        ax.imshow(r['psf'], cmap='inferno', origin='lower')
        ax.set_title(f"({r['x']:.0f},{r['y']:.0f})\nFWHM={r['fwhm']:.2f}px", fontsize=8)
    else:
        ax.axis('off')
    ax.tick_params(labelsize=5)
plt.suptitle(f'PSF grid — calexp {dataId} ({len(results)} valid, {len(failures)} invalid)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# FWHM map
fig, ax = plt.subplots(figsize=(8, 8))
v1, v2 = np.percentile(image, [5, 95])
ax.imshow(image, cmap='gray', origin='lower', vmin=v1, vmax=v2, alpha=0.5)
if results:
    sc = ax.scatter([r['x'] for r in results], [r['y'] for r in results],
                    c=fwhms, cmap='coolwarm', s=120, edgecolors='black', lw=0.5, zorder=5, label='Valid')
    plt.colorbar(sc, ax=ax, label='FWHM (px)', shrink=0.8)
    for r in results:
        ax.annotate(f"{r['fwhm']:.2f}", (r['x'], r['y']),
                    textcoords='offset points', xytext=(5, 5), fontsize=7, color='white')
if failures:
    ax.scatter([f['x'] for f in failures], [f['y'] for f in failures],
               c='red', marker='x', s=80, lw=1.5, label='Invalid')
ax.legend(loc='lower right')
ax.set_title(f'PSF FWHM — calexp {dataId}\nrange: [{min(fwhms):.3f}, {max(fwhms):.3f}] px')
ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
plt.tight_layout(); plt.show()

In [ ]:
print('=== PSF EXTRACTION RESULTS ===')
print(f'Image shape    : {image.shape}')
print(f'PSF type       : {type(psf).__name__}')
print(f'Pixel scale    : {PIXEL_SCALE:.4f} arcsec/px')
print(f'Grid test      : {len(results)}/{n_side**2} valid')
if results:
    print(f'FWHM mean      : {np.mean(fwhms):.4f} px = {np.mean(fwhms)*PIXEL_SCALE:.4f} arcsec')
    print(f'FWHM range     : [{min(fwhms):.4f}, {max(fwhms):.4f}] px')
    print(f'FWHM std       : {np.std(fwhms):.4f} px')
    if np.std(fwhms) < 0.01:
        print('  PSF is spatially constant (calexp PSF may be simpler than CoaddPsf)')
    else:
        print(f'  PSF varies by {(max(fwhms)-min(fwhms))/np.mean(fwhms)*100:.1f}% across the calexp')